# 改訂単体法

In [1]:
import numpy as np

### 最大係数規則

In [119]:
class RevisedSimplex:
    def __init__(self, A, b, c):
        self.A = A
        self.b = b
        self.c = c

        self.m, self.n = A.shape

        self.B_idx = list(range(self.n - self.m, self.n))
        self.N_idx = list(range(self.n - self.m))

    def solve(self):
        iteration = 0
        while True:
            # --- Step 1 ---
            B = self.A[:, self.B_idx] #基底行列
            N = self.A[:, self.N_idx] #非基底行列

            B_inv = np.linalg.inv(B) #基底行列の逆行列

            x_B = B_inv @ self.b  #初期の実行可能基底解
            b_bar = x_B
            # --- Step 2 ---
            c_B = self.c[self.B_idx]
            c_N = self.c[self.N_idx]

            c_bar = c_N - N.T @ B_inv.T @ c_B   # 被約費用の計算

            # --- Step 3 ---
            if np.all(c_bar <= 0): #c_barが全て非正であれば最適解
                x = np.zeros(self.n)
                x[self.B_idx] = b_bar
                return x, self.c @ x, iteration #最適解と最適値と反復回数を返して終了

            # c_barの中で最大のものを選び，対応する非基底変数を基底に入れる（最大係数規則）
            k = np.argmax(c_bar)
            entering = self.N_idx[k]

            # --- Step 4 ---
            a_k = self.A[:, entering]
            a_bar = B_inv @ a_k #N_barの非基底変数に対応する列ベクトル

            if np.all(a_bar <= 0):
               return None, None, iteration #非有界なので終了

            # --- Step 5 ---
            theta = []
            for i in range(self.m):
                if a_bar[i] > 0:
                    theta.append(b_bar[i] / a_bar[i])
                else:
                    theta.append(np.inf)

            i_star = np.argmin(theta)
            leaving = self.B_idx[i_star]

            # 基底更新
            self.B_idx[i_star] = entering
            self.N_idx[k] = leaving
            iteration += 1

            if iteration > 10000: 
                print("反復回数が上限に達しました。")
                return None, None, iteration

例1

In [106]:
A = np.array([[1, 1, 1, 0, 0],
              [1, 3, 0, 1, 0],
              [2, 1, 0, 0, 1]])
b = np.array([6, 12, 10])
c = np.array([1, 2, 0, 0, 0])

例2　非有界のとき

In [88]:
A = np.array([
    [1, -2, 1, 0],
    [-1, 1, 0, 1]])
b = np.array([4, 2])
c = np.array([2, 1, 0, 0])

例3　巡回に陥るとき

In [109]:
A = np.array([[0.5, -5.5, -2.5, 9, 1, 0, 0], 
              [0.5, -1.5, -0.5, 1, 0, 1, 0],
              [1, 0, 0, 0, 0, 0, 1]])
b = np.array([0, 0, 1])
c = np.array([10, -57, -9, -24, 0, 0, 0])

In [110]:
solver = RevisedSimplex(A, b, c)
x_opt, val, iteration = solver.solve()

反復回数が上限に達しました。


In [111]:
print("x* =", x_opt)
print("optimal value =", val)
print("iteration =", iteration)

x* = None
optimal value = None
iteration = 10001


### 最小添字規則

In [116]:
class RevisedSimplex:
    def __init__(self, A, b, c):
        self.A = A
        self.b = b
        self.c = c

        self.m, self.n = A.shape

        self.B_idx = list(range(self.n - self.m, self.n))
        self.N_idx = list(range(self.n - self.m))

    def solve(self):
        iteration = 0
        while True:
        # --- Step 1 ---
         B = self.A[:, self.B_idx]
         N = self.A[:, self.N_idx]

         B_inv = np.linalg.inv(B)
         b_bar = B_inv @ self.b

        # --- Step 2 ---
         c_B = self.c[self.B_idx]
         c_N = self.c[self.N_idx]

         c_bar = c_N - N.T @ B_inv.T @ c_B

        # --- Step 3 ---
         candidates = [j for j in range(len(c_bar)) if c_bar[j] > 0]

         if len(candidates) == 0:
            x = np.zeros(self.n)
            x[self.B_idx] = b_bar
            return x, self.c @ x, iteration

        # 最小添字規則（entering）
         k = min(candidates)
         entering = self.N_idx[k]

        # --- Step 4 ---
         a_k = self.A[:, entering]
         a_bar = B_inv @ a_k

         if np.all(a_bar <= 1e-10):
            return None, None, iteration  # 非有界

        # --- Step 5 ---
         theta = []
         for i in range(self.m):
            if a_bar[i] > 1e-10:
                theta.append(b_bar[i] / a_bar[i])
            else:
                theta.append(np.inf)

         theta_min = min(theta)

        # 最小添字規則（leaving）
         candidates = [i for i in range(self.m) if abs(theta[i] - theta_min) < 1e-12]
         i_star = min(candidates, key=lambda i: self.B_idx[i])

         leaving = self.B_idx[i_star]

        # 基底更新
         self.B_idx[i_star] = entering
         self.N_idx[k] = leaving
         iteration += 1

         if iteration > 10000:
            print("反復回数が上限に達しました。")
            return None, None, iteration
        

In [52]:
A = np.array([[0.5, -5.5, -2.5, 9, 1, 0, 0], 
              [0.5, -1.5, -0.5, 1, 0, 1, 0],
              [1, 0, 0, 0, 0, 0, 1]])
b = np.array([0, 0, 1])
c = np.array([10, -57, -9, -24, 0, 0, 0])

In [58]:
solver = RevisedSimplex(A, b, c)
x_opt, val, iteration = solver.solve()

反復回数が上限に達しました。


In [54]:
print("x* =", x_opt)
print("optimal value =", val)
print("iteration =", iteration)

x* = [1. 0. 1. 0. 2. 0. 0.]
optimal value = 1.0
iteration = 5


### クレー・ミンティの問題(4次元)

In [126]:
A= np.array([[1, 0, 0, 0, 1, 0, 0, 0],
             [20, 1, 0, 0, 0, 1, 0, 0],
             [200, 20, 1, 0, 0, 0, 1, 0],
             [2000, 200, 20, 1, 0, 0, 0, 1]])
b = np.array([1, 100, 10000, 1000000])
c = np.array([1000, 100, 10, 1, 0, 0, 0, 0])

In [127]:
solver = RevisedSimplex(A, b, c)
x_opt, val, iteration = solver.solve()
print("x* =", x_opt)
print("optimal value =", val)
print("iteration=", iteration)

x* = [0.e+00 0.e+00 0.e+00 1.e+06 1.e+00 1.e+02 1.e+04 0.e+00]
optimal value = 1000000.0
iteration= 15
